In [1]:
import io
import json
import os
from pathlib import Path

import pandas as pd
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Análise por período (dados baixados pelo `src/00-dowload-data.R`)

Seção independente das anteriores: pode ser rodada sozinha, sem login no Drive.

Hierarquia dos dados: **contratações > itens > medicamentos > resultados**.

1. carrega contratações, itens, medicamentos e resultados de **um mesmo período** em `tmp/`;
2. busca nas contratações os CNPJs-alvo (por prefixo, ex.: `^18729020` pega todas as filiais dessa raiz);
3. junta as contratações encontradas aos seus itens, marcando quais itens são medicamentos;
4. junta os resultados (que só existem para itens de medicamentos);
5. salva tudo em `output/<período>/`, ex.: `output/2025-04-QUINZENA-1/`.

**Chave de junção:** contratações têm `data.numeroControlePNCP` (`<cnpj>-1-<sequencial 6 dígitos>/<ano>`);
itens e medicamentos só têm `endpoint` (`.../orgaos/<cnpj>/compras/<ano>/<sequencial>/itens`), de onde a
mesma chave é reconstruída; resultados trazem a chave em `numeroControlePNCPCompra`.

### Configuração

In [112]:
import re
from pathlib import Path

import pandas as pd


def encontra_raiz_repositorio(inicio: Path) -> Path:
    for pasta in [inicio, *inicio.parents]:
        if (pasta / ".git").exists():
            return pasta
    raise FileNotFoundError("Raiz do repositório não encontrada a partir de " + str(inicio))


REPO_ROOT = encontra_raiz_repositorio(Path.cwd().resolve())
TASK_DIR = REPO_ROOT / "tasks" / "analise-unidades-compradoras"
TMP_DIR = TASK_DIR / "tmp"
OUTPUT_DIR = TASK_DIR / "output"

# período analisado: pasta em tmp/ no formato "Coletas - <ano>/<mês> - <Nome>/<quinzena>"
PERIODO_DIR = TMP_DIR / "Coletas - 2026" / "6 - Junho" / "QUINZENA-2"

# CNPJs-alvo, tratados como prefixo: um CNPJ completo (14 dígitos) busca só ele,
# uma raiz (ex.: 8 dígitos) busca todas as filiais. Aceita com ou sem pontuação.
CNPJS_ALVO = {
    "unidade01": "25.089.137/0001-95",
    "unidade02": "16.723.250/0001-90",
    "unidade03": "18.729.020",
}

In [113]:
def limpa_cnpj(cnpj: str) -> str:
    return re.sub(r"\D", "", cnpj)


def nome_periodo(pasta: Path) -> str:
    """'Coletas - 2025/4 - Abril/QUINZENA-1' -> '2025-04-QUINZENA-1'."""
    ano = re.search(r"\d{4}", pasta.parent.parent.name).group()
    mes = int(re.match(r"\d+", pasta.parent.name).group())
    return f"{ano}-{mes:02d}-{pasta.name}"


def chave_compra(endpoint: pd.Series) -> pd.Series:
    """Reconstrói o numeroControlePNCP a partir do endpoint de itens.

    O '-1-' é fixo: todas as contratações coletadas são do tipo compra (conferido nos dados).
    """
    partes = endpoint.str.extract(r"/orgaos/(\d{14})/compras/(\d{4})/(\d+)/itens")
    return partes[0] + "-1-" + partes[2].str.zfill(6) + "/" + partes[1]


PERIODO = nome_periodo(PERIODO_DIR)
PREFIXOS_ALVO = {nome: limpa_cnpj(cnpj) for nome, cnpj in CNPJS_ALVO.items()}
REGEX_CNPJ = "^(?:" + "|".join(PREFIXOS_ALVO.values()) + ")"
print(PERIODO, REGEX_CNPJ)

2026-06-QUINZENA-2 ^(?:25089137000195|16723250000190|18729020)


### 1. Leitura dos dados do período

Tudo é lido como texto (`dtype=str`) para não perder zeros à esquerda de CNPJs e códigos.
A coluna `embedding` dos medicamentos é descartada (pesada e desnecessária aqui).
Se houver `itens-medicamentos-resultados-recoleta.csv`, ele é somado aos resultados e,
em caso de duplicata, prevalece a recoleta.

In [114]:
contratacoes = pd.read_csv(PERIODO_DIR / "contratacoes.csv", dtype=str)
itens = pd.read_csv(PERIODO_DIR / "itens.csv", dtype=str)
medicamentos = pd.read_csv(PERIODO_DIR / "itens-medicamentos.csv", dtype=str, usecols=lambda c: c != "embedding")

arquivos_resultados = [PERIODO_DIR / "itens-medicamentos-resultados.csv"]
if (PERIODO_DIR / "itens-medicamentos-resultados-recoleta.csv").exists():
    arquivos_resultados.append(PERIODO_DIR / "itens-medicamentos-resultados-recoleta.csv")
resultados = (
    pd.concat([pd.read_csv(f, dtype=str) for f in arquivos_resultados], ignore_index=True)
    .drop_duplicates(["numeroControlePNCPCompra", "numeroItem", "sequencialResultado"], keep="last")
)

itens["numeroControlePNCP"] = chave_compra(itens["endpoint"])
medicamentos["numeroControlePNCP"] = chave_compra(medicamentos["endpoint"])

pd.DataFrame(
    {
        "linhas": [len(contratacoes), len(itens), len(medicamentos), len(resultados)],
        "arquivos": ["contratacoes.csv", "itens.csv", "itens-medicamentos.csv", ", ".join(f.name for f in arquivos_resultados)],
    },
    index=["contratações", "itens", "medicamentos", "resultados"],
)

,linhas,arquivos
contratações,73584,contratacoes.csv
itens,243685,itens.csv
medicamentos,8369,itens-medicamentos.csv
resultados,5370,itens-medicamentos-resultados.csv


### 2. Contratações dos CNPJs-alvo

Busca pelo CNPJ do órgão responsável (`data.orgaoEntidade.cnpj`). A coluna `unidade_alvo`
guarda o CNPJ (ou raiz) de `CNPJS_ALVO`, só com dígitos, que casou com a contratação.

In [115]:
def identifica_alvo(cnpj: str) -> str | None:
    for nome, prefixo in PREFIXOS_ALVO.items():
        if isinstance(cnpj, str) and cnpj.startswith(prefixo):
            return prefixo
    return None


contratacoes_alvo = contratacoes[
    contratacoes["data.orgaoEntidade.cnpj"].str.contains(REGEX_CNPJ, regex=True, na=False)
].copy()
contratacoes_alvo.insert(0, "unidade_alvo", contratacoes_alvo["data.orgaoEntidade.cnpj"].map(identifica_alvo))

print(f"{len(contratacoes_alvo)} de {len(contratacoes)} contratações do período são dos CNPJs-alvo")

nao_encontrados = {n for n, p in PREFIXOS_ALVO.items() if p not in set(contratacoes_alvo["unidade_alvo"])}
if nao_encontrados:
    print("Sem contratações neste período:", ", ".join(f"{n} ({PREFIXOS_ALVO[n]})" for n in sorted(nao_encontrados)))

(
    contratacoes_alvo.groupby(
        ["unidade_alvo", "data.orgaoEntidade.cnpj", "data.orgaoEntidade.razaoSocial", "data.unidadeOrgao.nomeUnidade"],
        dropna=False,
    )
    .agg(contratacoes=("data.numeroControlePNCP", "nunique"))
    .reset_index()
)

0 de 73584 contratações do período são dos CNPJs-alvo
Sem contratações neste período: unidade01 (25089137000195), unidade02 (16723250000190), unidade03 (18729020)


,unidade_alvo,data.orgaoEntidade.cnpj,data.orgaoEntidade.razaoSocial,data.unidadeOrgao.nomeUnidade,contratacoes


### 3. Itens das contratações encontradas

Todos os itens das contratações-alvo, com os dados da contratação ao lado. Itens que também estão em
`itens-medicamentos.csv` ganham `medicamento = True` e as colunas do classificador (`codigo_pdm`, `codigo_br`, `similaridade`).

In [116]:
COLUNAS_CONTRATACAO = [
    "unidade_alvo",
    "data.numeroControlePNCP",
    "data.orgaoEntidade.cnpj",
    "data.orgaoEntidade.razaoSocial",
    "data.unidadeOrgao.codigoUnidade",
    "data.unidadeOrgao.nomeUnidade",
    "data.unidadeOrgao.municipioNome",
    "data.unidadeOrgao.ufSigla",
    "data.modalidadeNome",
    "data.situacaoCompraNome",
    "data.dataPublicacaoPncp",
    "data.objetoCompra",
]
COLUNAS_CLASSIFICADOR = ["codigo_pdm", "codigo_br", "similaridade"]

itens_alvo = (
    contratacoes_alvo[COLUNAS_CONTRATACAO]
    .rename(columns={"data.numeroControlePNCP": "numeroControlePNCP"})
    .merge(itens, on="numeroControlePNCP", how="inner", validate="1:m")
    .merge(
        medicamentos[["numeroControlePNCP", "numeroItem", *COLUNAS_CLASSIFICADOR]].assign(medicamento=True),
        on=["numeroControlePNCP", "numeroItem"],
        how="left",
        validate="1:1",
    )
)
itens_alvo["medicamento"] = itens_alvo["medicamento"].fillna(False).astype(bool)

sem_itens = set(contratacoes_alvo["data.numeroControlePNCP"]) - set(itens_alvo["numeroControlePNCP"])
print(f"{len(itens_alvo)} itens ({itens_alvo['medicamento'].sum()} medicamentos) "
      f"em {itens_alvo['numeroControlePNCP'].nunique()} contratações")
if sem_itens:
    print(f"{len(sem_itens)} contratação(ões) sem itens no período:", ", ".join(sorted(sem_itens)))

itens_alvo.groupby(["unidade_alvo", "medicamento"]).size().unstack(fill_value=0)

0 itens (0 medicamentos) em 0 contratações


medicamento
unidade_alvo


### 4. Resultados dos medicamentos

Resultados só são coletados para itens de medicamentos. Cada item pode ter mais de um resultado
(um por fornecedor homologado), então a tabela tem uma linha por item × resultado; medicamentos sem
resultado aparecem uma vez, com as colunas `resultado.*` vazias. As colunas dos resultados
recebem o prefixo `resultado.` para não colidir com as dos itens (`dataInclusao`, `endpoint` etc.).

In [117]:
resultados_prefixados = resultados.rename(columns={"numeroControlePNCPCompra": "numeroControlePNCP"}).set_index(
    ["numeroControlePNCP", "numeroItem"]
).add_prefix("resultado.").reset_index()

medicamentos_resultados = itens_alvo[itens_alvo["medicamento"]].merge(
    resultados_prefixados, on=["numeroControlePNCP", "numeroItem"], how="left", validate="1:m"
)

com_resultado = medicamentos_resultados["resultado.sequencialResultado"].notna()
print(f"{medicamentos_resultados.loc[com_resultado, ['numeroControlePNCP', 'numeroItem']].drop_duplicates().shape[0]} "
      f"de {itens_alvo['medicamento'].sum()} medicamentos têm resultado ({com_resultado.sum()} linhas de resultado)")

medicamentos_resultados[
    [
        "unidade_alvo",
        "numeroControlePNCP",
        "numeroItem",
        "descricao",
        "codigo_br",
        "quantidade",
        "valorUnitarioEstimado",
        "resultado.nomeRazaoSocialFornecedor",
        "resultado.quantidadeHomologada",
        "resultado.valorUnitarioHomologado",
        "resultado.situacaoCompraItemResultadoNome",
    ]
]

0 de 0 medicamentos têm resultado (0 linhas de resultado)


,unidade_alvo,numeroControlePNCP,numeroItem,descricao,codigo_br,quantidade,valorUnitarioEstimado,resultado.nomeRazaoSocialFornecedor,resultado.quantidadeHomologada,resultado.valorUnitarioHomologado,resultado.situacaoCompraItemResultadoNome


### 5. Salva os resultados do período

Em `output/<período>/`:

- `contratacoes.csv`: contratações dos CNPJs-alvo (todas as colunas);
- `itens.csv`: itens dessas contratações, com dados da contratação e a marcação `medicamento`;
- `medicamentos-resultados.csv`: medicamentos com seus resultados (uma linha por item × resultado).

In [118]:
pasta_saida = OUTPUT_DIR / PERIODO
pasta_saida.mkdir(parents=True, exist_ok=True)

saidas = {
    "contratacoes.csv": contratacoes_alvo,
    "itens.csv": itens_alvo,
    "medicamentos-resultados.csv": medicamentos_resultados,
}
for nome_arquivo, df in saidas.items():
    df.to_csv(pasta_saida / nome_arquivo, index=False)
    print(f"{pasta_saida.relative_to(REPO_ROOT) / nome_arquivo}: {len(df)} linhas")

tasks\analise-unidades-compradoras\output\2026-06-QUINZENA-2\contratacoes.csv: 0 linhas
tasks\analise-unidades-compradoras\output\2026-06-QUINZENA-2\itens.csv: 0 linhas
tasks\analise-unidades-compradoras\output\2026-06-QUINZENA-2\medicamentos-resultados.csv: 0 linhas
